In [1]:
# Example for using the model registry builders
#
# The purpose of these builders is to simplify the tedium
# of writing out the model_registry dictionaries by hand,
# especially when a lot of the state and param variables
# may simply be inferred from the model dataclass

# Package imports
from dataclasses import dataclass
from sango.model import Neuron, Synapse, PSP, shared

# These models are taken from the example_brian.ipynb notebook
from sango.backend import SimBrian, BrianRegistry

# Define new model classes
@dataclass
class CUBA(Neuron):
    model: str = 'CUBA'
    v: float = 0.0
    ge: float = 0.0
    gi: float = 0.0
    Vt: float = shared(10.0)
    Vr: float = shared(0.0)
    El: float = shared(5.0)
    taum: float = shared(20.0)
    taue: float = shared(5.0)
    taui: float = shared(10.0)
    tref: float = shared(5.0)

@dataclass
class PSPe(PSP):
    model: str = 'PSPe'

@dataclass
class PSPi(PSP):
    model: str = 'PSPi'

# Define model registry with how the new models map to the backend
brian_registry = {'CUBA': {'graph_type': 'neuron',
                           'model_eqs' : '''
                                         dv/dt  = (ge+gi-(v-El))/taum : 1 (unless refractory)
                                         dge/dt = -ge/taue : 1
                                         dgi/dt = -gi/taui : 1
                                         ''',
                           'method'    : 'exact',
                           'threshold' : 'v>Vt',
                           'reset'     : 'v=Vr',
                           'refractory': 'tref',
                           'events'    : {},
                           'state': {'v':      {'mapfrom': 'v',     'default': -60.0},
                                     'ge':     {'mapfrom': 'ge',    'default': 0.0},
                                     'gi':     {'mapfrom': 'gi',    'default': 0.0}},
                           'param': {'Vt':     {'mapfrom': 'Vt',    'default': -50.0},
                                     'Vr':     {'mapfrom': 'Vr',    'default': -60.0},
                                     'El':     {'mapfrom': 'El',    'default': -49.0},
                                     'taum':   {'mapfrom': 'taum',  'default': 20.0, 'unit': 'ms'},
                                     'taue':   {'mapfrom': 'taue',  'default': 5.0,  'unit': 'ms'},
                                     'taui':   {'mapfrom': 'taui',  'default': 10.0, 'unit': 'ms'},
                                     'tref':   {'mapfrom': 'tref',  'default': 5.0,  'unit': 'ms'}}},
                  'PSPe': {'graph_type': 'synapse',
                           'model_eqs' : 'weight : 1',
                           'on_pre'    : 'ge+=weight',
                           'state': {'delay':    {'mapfrom': 'delay',  'default': 1.0, 'unit': 'ms'},
                                     'weight':   {'mapfrom': 'weight', 'default': 1.0}}},
                  'PSPi': {'graph_type': 'synapse',
                           'model_eqs' : 'weight : 1',
                           'on_pre'    : 'gi+=weight',
                           'state': {'delay':    {'mapfrom': 'delay',  'default': 1.0, 'unit': 'ms'},
                                     'weight':   {'mapfrom': 'weight', 'default': 1.0}}}
                 }

# Model registry builder with fluent API
# wrapped in parenthesis for multi-line
new_brian_registry = BrianRegistry()
(new_brian_registry.register(CUBA)
    .model_eqs('''
    dv/dt  = (ge+gi-(v-El))/taum : 1 (unless refractory)
    dge/dt = -ge/taue : 1
    dgi/dt = -gi/taui : 1
    ''')
    .threshold('v>Vt')
    .reset('v=Vr')
    .refractory('tref')
    .clocked(['taum', 'taue', 'taui', 'tref']));
(new_brian_registry.register(PSPe)
    .model_eqs('weight : 1')
    .on_pre('ge+=weight'));
(new_brian_registry.register(PSPi)
    .model_eqs('weight : 1')
    .on_pre('gi+=weight'));

# Convert builder to dict, comparison done visually
# because the multi-line equations are hard to match
new_brian_registry.to_dict()

{'CUBA': {'graph_type': 'neuron',
  'model_eqs': '\n    dv/dt  = (ge+gi-(v-El))/taum : 1 (unless refractory)\n    dge/dt = -ge/taue : 1\n    dgi/dt = -gi/taui : 1\n    ',
  'method': 'exact',
  'threshold': 'v>Vt',
  'reset': 'v=Vr',
  'refractory': 'tref',
  'events': {},
  'state': {'v': {'mapfrom': 'v', 'default': 0.0},
   'ge': {'mapfrom': 'ge', 'default': 0.0},
   'gi': {'mapfrom': 'gi', 'default': 0.0}},
  'param': {'Vt': {'mapfrom': 'Vt', 'default': 10.0},
   'Vr': {'mapfrom': 'Vr', 'default': 0.0},
   'El': {'mapfrom': 'El', 'default': 5.0},
   'taum': {'mapfrom': 'taum', 'default': 20.0, 'unit': 'ms'},
   'taue': {'mapfrom': 'taue', 'default': 5.0, 'unit': 'ms'},
   'taui': {'mapfrom': 'taui', 'default': 10.0, 'unit': 'ms'},
   'tref': {'mapfrom': 'tref', 'default': 5.0, 'unit': 'ms'}}},
 'PSPe': {'graph_type': 'synapse',
  'model_eqs': 'weight : 1',
  'on_pre': 'ge+=weight',
  'state': {'delay': {'mapfrom': 'delay', 'default': 1.0},
   'weight': {'mapfrom': 'weight', 'default

In [2]:
# These models are taken from the example_polynet.ipynb notebook
from sango.backend import SimStacs, StacsRegistry

# Cross-referenced models from STACS
@dataclass
class Izhi(Neuron):
    model: str = 'Izhi'
    a: float = shared(0.02)
    b: float = shared(0.2)
    c: float = shared(-65.0)
    d: float = shared(8.0)
    thal_rate: float = shared(1.0)
    thal_ampl: float = shared(20.0)
    psn_rate: float = shared(0.0)
    v: float = -65.0
    u: float = -13.0

@dataclass
class STDP(Synapse):
    model: str = 'STDP'
    w_max: float = shared(10.0)
    w_min: float = shared(0.0)
    A_pos: float = shared(0.1)
    A_neg: float = shared(-0.12)
    tau_pos: float = shared(20.0)
    tau_neg: float = shared(20.0)
    wd_const: float = shared(0.01)
    wd_lambda: float = shared(0.9)
    delay: float = 1.0
    weight: float = 1.0

# Matching class information in STACS models
polynet_registry = {'Izhi': {'graph_type': 'node',
                             'model_type': 'izhi_neuron_clamp',
                             'param': {'a':          {'mapfrom': 'a',         'default': 0.02},
                                       'b':          {'mapfrom': 'b',         'default': 0.2},
                                       'c':          {'mapfrom': 'c',         'default': -65.0},
                                       'd':          {'mapfrom': 'd',         'default': 8.0},
                                       'thal_rate':  {'mapfrom': 'thal_rate', 'default': 1.0},
                                       'thal_ampl':  {'mapfrom': 'thal_ampl', 'default': 20.0},
                                       'psn_rate':   {'mapfrom': 'psn_rate',  'default': 0.0}},
                             'state': {'v':          {'mapfrom': 'v',         'default': -65.0},
                                       'u':          {'mapfrom': 'u',         'default': -13.0},
                                       'I_syn':      {'mapfrom': None,        'default': 0.0},
                                       'I_app':      {'mapfrom': None,        'default': 0.0},
                                       'I_clamp':    {'mapfrom': None,        'default': 0.0}}},
                    'STDP': {'graph_type': 'edge',
                             'model_type': 'stdp_synapse',
                             'param': {'w_max':      {'mapfrom': 'w_max',     'default': 10.0},
                                       'w_min':      {'mapfrom': 'w_min',     'default': 0.0},
                                       'A_pos':      {'mapfrom': 'A_pos',     'default': 0.1},
                                       'A_neg':      {'mapfrom': 'A_neg',     'default': -0.12},
                                       'tau_pos':    {'mapfrom': 'tau_pos',   'default': 20.0},
                                       'tau_neg':    {'mapfrom': 'tau_neg',   'default': 20.0},
                                       'wd_const':   {'mapfrom': 'wd_const',  'default': 0.01},
                                       'wd_lambda':  {'mapfrom': 'wd_lambda', 'default': 0.9},
                                       't_update':   {'mapfrom': None,        'default': 1000.0}},
                             'state': {'delay':      {'mapfrom': 'delay',     'default': 1.0, 'unit': 'tick'},
                                       'weight':     {'mapfrom': 'weight',    'default': 1.0},
                                       'wdelta':     {'mapfrom': None,        'default': 0.0},
                                       'ptrace':     {'mapfrom': None,        'default': 0.0},
                                       'ntrace':     {'mapfrom': None,        'default': 0.0},
                                       'ptlast':     {'mapfrom': None,        'default': 0.0, 'unit': 'tick'},
                                       'ntlast':     {'mapfrom': None,        'default': 0.0, 'unit': 'tick'}}}
                    }

# Model registry builder with fluent API
# wrapped in parenthesis for multi-line
new_polynet_registry = StacsRegistry()
(new_polynet_registry.register(Izhi)
    .model_type('izhi_neuron_clamp')
    .extra_states(['I_syn', 'I_app', 'I_clamp']));
(new_polynet_registry.register(STDP)
    .model_type('stdp_synapse')
    .extra_param('t_update', default=1000.0)
    .extra_states(['wdelta', 'ptrace', 'ntrace'])
    .extra_states({'ptlast': {'unit': 'tick'},
                   'ntlast': {'unit': 'tick'}})
    .clocked('delay'));

# There are no multi-line strings in this one
print(new_polynet_registry.to_dict() == polynet_registry)

True
